# Financial News Sentiment Prediction using Deep Learning & BERT

**Objective:** Build a sentiment classification system that labels finance-related tweets as **Bearish**, **Bullish**, or **Neutral** using deep learning.

**Dataset:** [Twitter Financial News Sentiment](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment) from Hugging Face

**Models:**
- Embedding + Simple RNN (baseline)
- Embedding + LSTM
- Embedding + GRU
- Fine-tuned BERT/FinBERT (optional improvement)

**Evaluation Metrics:** Overall Accuracy, Macro F1-Score, Class-wise Precision & Recall, Confusion Matrix

## 1. Setup and Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure src is on the path
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.data_loading import (load_financial_sentiment_data, validate_split_counts,
                              get_class_distribution, get_basic_stats,
                              LABEL_MAP, NUM_CLASSES)
from src.preprocessing import (clean_tweet, build_vocabulary, create_dataloaders)
from src.models import get_rnn_model, load_bert_model
from src.training import train_rnn_model, train_bert_model
from src.evaluation import (evaluate_rnn_model, evaluate_bert_model,
                            compute_metrics, plot_confusion_matrix,
                            plot_training_history, compare_models)
from src.inference import predict_paragraph

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Load and Understand the Dataset

We use the Hugging Face `zeroshot/twitter-financial-news-sentiment` dataset.
- **Labels:** LABEL_0 → Bearish, LABEL_1 → Bullish, LABEL_2 → Neutral
- **Splits:** 9,938 training / 2,486 validation instances

In [ ]:
# Load the dataset
train_df, val_df = load_financial_sentiment_data()
split_counts = validate_split_counts(train_df, val_df)

print(f'Training samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'\nColumns: {list(train_df.columns)}')
print(f'Label mapping: {LABEL_MAP}')

# Preview data
train_df.head(10)

In [ ]:
# Check basic statistics
get_basic_stats(train_df, 'Training')
get_basic_stats(val_df, 'Validation')

In [ ]:
# Class distribution
train_dist = get_class_distribution(train_df, 'Training')
val_dist = get_class_distribution(val_df, 'Validation')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#2ecc71', '#3498db']  # Red for Bearish, Green for Bullish, Blue for Neutral

for ax, dist_df, title in zip(axes, [train_dist, val_dist], ['Training', 'Validation']):
    bars = ax.bar(dist_df['Label_Name'], dist_df['Count'], color=colors)
    ax.set_title(f'{title} Set Class Distribution')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    for bar, pct in zip(bars, dist_df['Percentage']):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Preprocess Tweets

**Steps:**
- Clean text: remove URLs, @mentions, extra spaces; keep $tickers and financial terms
- Tokenize and build vocabulary
- Prepare padded train/validation tensors

In [ ]:
# Clean tweets
train_df['clean_text'] = train_df['text'].apply(clean_tweet)
val_df['clean_text'] = val_df['text'].apply(clean_tweet)

# Show before/after cleaning examples
print('=== Cleaning Examples ===')
for i in range(5):
    print(f'\nOriginal: {train_df["text"].iloc[i]}')
    print(f'Cleaned:  {train_df["clean_text"].iloc[i]}')

In [ ]:
# Build vocabulary from training data ONLY (no data leakage)
word2idx, idx2word, vocab_size = build_vocabulary(
    train_df['clean_text'].tolist(), min_freq=2
)
print(f'\nSample vocabulary entries: {list(word2idx.items())[:10]}')

In [ ]:
# Prepare DataLoaders
MAX_LEN = 64
BATCH_SIZE = 64

train_loader, val_loader = create_dataloaders(
    train_texts=train_df['clean_text'].tolist(),
    train_labels=train_df['label'].tolist(),
    val_texts=val_df['clean_text'].tolist(),
    val_labels=val_df['label'].tolist(),
    word2idx=word2idx,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE
)

# Verify a batch
sample_texts, sample_labels = next(iter(train_loader))
print(f'Batch text shape: {sample_texts.shape}')
print(f'Batch label shape: {sample_labels.shape}')
print(f'Label values in batch: {sample_labels.unique().tolist()}')

## 4. Build and Train Baseline Deep Learning Models

We implement three RNN-based architectures:
1. **Embedding + Simple RNN**
2. **Embedding + LSTM**
3. **Embedding + GRU**

All models are trained with early stopping to prevent overfitting.

In [ ]:
# Hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.3
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15
PATIENCE = 3

# Store results for comparison
all_results = {}
all_histories = {}
trained_models = {}

In [ ]:
# Calculate class weights inversely proportional to class frequencies
class_counts = train_df['label'].value_counts().sort_index()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)  # Normalize
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float).to(device)
print("Class weights:", class_weights_tensor)

### 4.1 Embedding + Simple RNN

In [ ]:
# Build Simple RNN model
simple_rnn = get_rnn_model(
    model_type='simple_rnn',
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)
print(simple_rnn)
print(f'\nTotal parameters: {sum(p.numel() for p in simple_rnn.parameters()):,}')

In [ ]:
print('=== Training Simple RNN ===')
simple_rnn, rnn_history = train_rnn_model(
    simple_rnn, train_loader, val_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    patience=PATIENCE, device=device, class_weights=class_weights_tensor
)
all_histories['Simple RNN'] = rnn_history
trained_models['Simple RNN'] = simple_rnn

In [ ]:
# Evaluate Simple RNN
print('=== Simple RNN Evaluation ===')
rnn_preds, rnn_labels, rnn_probs = evaluate_rnn_model(simple_rnn, val_loader, device)
rnn_metrics = compute_metrics(rnn_labels, rnn_preds)
all_results['Simple RNN'] = rnn_metrics

# Confusion Matrix
fig = plot_confusion_matrix(rnn_labels, rnn_preds, 'Simple RNN - Confusion Matrix')
plt.show()

# Training curves
fig = plot_training_history(rnn_history, 'Simple RNN')
plt.show()

### 4.2 Embedding + LSTM

In [ ]:
# Build LSTM model
lstm_model = get_rnn_model(
    model_type='lstm',
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)
print(lstm_model)
print(f'\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

In [ ]:
print('=== Training LSTM ===')
lstm_model, lstm_history = train_rnn_model(
    lstm_model, train_loader, val_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    patience=PATIENCE, device=device, class_weights=class_weights_tensor
)
all_histories['LSTM'] = lstm_history
trained_models['LSTM'] = lstm_model

In [ ]:
# Evaluate LSTM
print('=== LSTM Evaluation ===')
lstm_preds, lstm_labels, lstm_probs = evaluate_rnn_model(lstm_model, val_loader, device)
lstm_metrics = compute_metrics(lstm_labels, lstm_preds)
all_results['LSTM'] = lstm_metrics

fig = plot_confusion_matrix(lstm_labels, lstm_preds, 'LSTM - Confusion Matrix')
plt.show()

fig = plot_training_history(lstm_history, 'LSTM')
plt.show()

### 4.3 Embedding + GRU

In [ ]:
# Build GRU model
gru_model = get_rnn_model(
    model_type='gru',
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)
print(gru_model)
print(f'\nTotal parameters: {sum(p.numel() for p in gru_model.parameters()):,}')

In [ ]:
print('=== Training GRU ===')
gru_model, gru_history = train_rnn_model(
    gru_model, train_loader, val_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    patience=PATIENCE, device=device, class_weights=class_weights_tensor
)
all_histories['GRU'] = gru_history
trained_models['GRU'] = gru_model

In [ ]:
# Evaluate GRU
print('=== GRU Evaluation ===')
gru_preds, gru_labels, gru_probs = evaluate_rnn_model(gru_model, val_loader, device)
gru_metrics = compute_metrics(gru_labels, gru_preds)
all_results['GRU'] = gru_metrics

fig = plot_confusion_matrix(gru_labels, gru_preds, 'GRU - Confusion Matrix')
plt.show()

fig = plot_training_history(gru_history, 'GRU')
plt.show()

## 5. (Optional) Improve with BERT

Fine-tune a pre-trained FinBERT model for 3-class sentiment classification on the same dataset.

In [ ]:
# Load pre-trained BERT/FinBERT model
print('Loading FinBERT model...')
bert_model, bert_tokenizer = load_bert_model(
    model_name='ahmedrachid/FinancialBERT-Sentiment-Analysis',
    num_labels=NUM_CLASSES
)
print(f'BERT parameters: {sum(p.numel() for p in bert_model.parameters()):,}')

In [ ]:
# Fine-tune BERT
print('=== Fine-tuning BERT ===')
bert_model, bert_history = train_bert_model(
    bert_model, bert_tokenizer,
    train_texts=train_df['text'].tolist(),
    train_labels=train_df['label'].tolist(),
    val_texts=val_df['text'].tolist(),
    val_labels=val_df['label'].tolist(),
    num_epochs=2,
    learning_rate=2e-5,
    batch_size=32,
    max_len=128,
    patience=2,
    device=device
)
all_histories['BERT (FinBERT)'] = bert_history

In [ ]:
# Evaluate BERT
print('=== BERT Evaluation ===')
bert_preds, bert_labels, bert_probs = evaluate_bert_model(
    bert_model, bert_tokenizer,
    texts=val_df['text'].tolist(),
    labels=val_df['label'].tolist(),
    max_len=128, batch_size=16, device=device
)
bert_metrics = compute_metrics(bert_labels, bert_preds)
all_results['BERT (FinBERT)'] = bert_metrics

fig = plot_confusion_matrix(bert_labels, bert_preds, 'BERT (FinBERT) - Confusion Matrix')
plt.show()


## 6. Model Comparison and Analysis

Compare all models across accuracy, macro F1-score, and class-wise metrics.

In [ ]:
# Comparison table
comparison_data = []
for model_name, metrics in all_results.items():
    row = {
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Macro F1': f"{metrics['macro_f1']:.4f}",
        'Bearish Precision': f"{metrics['class_precision']['Bearish']:.4f}",
        'Bearish Recall': f"{metrics['class_recall']['Bearish']:.4f}",
        'Bullish Precision': f"{metrics['class_precision']['Bullish']:.4f}",
        'Bullish Recall': f"{metrics['class_recall']['Bullish']:.4f}",
        'Neutral Precision': f"{metrics['class_precision']['Neutral']:.4f}",
        'Neutral Recall': f"{metrics['class_recall']['Neutral']:.4f}",
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print('\n=== Model Comparison ===')
comparison_df

In [ ]:
# Visual comparison
fig = compare_models(all_results)
plt.show()

In [ ]:
# Identify best RNN model and compare with BERT
rnn_models = {k: v for k, v in all_results.items() if 'BERT' not in k}
best_rnn_name = max(rnn_models, key=lambda k: rnn_models[k]['macro_f1'])
best_rnn_f1 = rnn_models[best_rnn_name]['macro_f1']

print(f'\n=== Analysis ===')
print(f'Best RNN variant: {best_rnn_name} (Macro F1: {best_rnn_f1:.4f})')

if 'BERT (FinBERT)' in all_results:
    bert_f1 = all_results['BERT (FinBERT)']['macro_f1']
    improvement = (bert_f1 - best_rnn_f1) / best_rnn_f1 * 100
    print(f'BERT Macro F1: {bert_f1:.4f}')
    print(f'F1 improvement from best RNN to BERT: {improvement:+.1f}%')
    print(f'\nBERT\'s pre-trained language understanding and contextual embeddings')
    print(f'allow it to capture nuanced financial sentiment that simple RNN architectures miss.')

## 7. Save Models

In [ ]:
import json

# Create models directory
os.makedirs('models/saved_models', exist_ok=True)

# Save all RNN models (Simple RNN, LSTM, GRU)
rnn_model_names = ['Simple RNN', 'LSTM', 'GRU']
for name in rnn_model_names:
    model = trained_models[name]
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_type': name.lower().replace(' ', '_'),
        'vocab_size': vocab_size,
        'embed_dim': EMBED_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_classes': NUM_CLASSES,
        'max_len': MAX_LEN
    }, f"models/saved_models/{name.lower().replace(' ', '_')}_model.pt")
    print(f"Saved {name} model")

# Save the best RNN model for backward compatibility
best_rnn_model = trained_models[best_rnn_name]
torch.save({
    'model_state_dict': best_rnn_model.state_dict(),
    'model_type': best_rnn_name.lower().replace(' ', '_'),
    'vocab_size': vocab_size,
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_classes': NUM_CLASSES,
    'max_len': MAX_LEN
}, 'models/saved_models/best_rnn_model.pt')
print(f'Saved best RNN model ({best_rnn_name})')

# Save vocabulary
with open('models/saved_models/word2idx.json', 'w') as f:
    json.dump(word2idx, f)
print('Saved vocabulary')

# Save BERT model (if trained)
if 'BERT (FinBERT)' in all_results:
    # Save to 'bert_finetuned' so the Streamlit app can find it
    bert_save_path = 'models/saved_models/bert_finetuned'
    bert_model.save_pretrained(bert_save_path)
    bert_tokenizer.save_pretrained(bert_save_path)
    print(f'Saved fine-tuned BERT model to {bert_save_path}')

print('\nAll models saved successfully!')


## 8. Automated Inference Script

The final output: an automated script that takes a raw financial paragraph and outputs the sentiment.

In [ ]:
# Test with sample financial texts
test_texts = [
    "$AAPL Apple stock surges to all-time high after strong earnings report, revenue beats expectations",
    "$TSLA Tesla shares plummet 15% amid production delays and growing competition in EV market",
    "The Federal Reserve announced it will maintain current interest rates through next quarter",
    "$AMZN Amazon reports mixed results with cloud growth slowing but retail improving",
    "Market crash fears grow as inflation data exceeds forecasts, investors flee to bonds"
]

print('=== Automated Sentiment Prediction ===')
print(f'Using model: {best_rnn_name}\n')

for text in test_texts:
    result = predict_paragraph(
        text=text,
        model=best_rnn_model,
        word2idx=word2idx,
        model_type='rnn',
        max_len=MAX_LEN,
        device=device
    )
    print('---')

In [ ]:
# Also test with BERT if available
if 'BERT (FinBERT)' in all_results:
    print('=== BERT Sentiment Prediction ===\n')
    for text in test_texts:
        result = predict_paragraph(
            text=text,
            model=bert_model,
            tokenizer=bert_tokenizer,
            model_type='bert',
            max_len=128,
            device=device
        )
        print('---')

## 9. Generate Comparison Report

In [ ]:
from fpdf import FPDF

os.makedirs('reports', exist_ok=True)

# Save comparison chart
fig = compare_models(all_results)
fig.savefig('reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

# Save best confusion matrices
for name, (preds, labels) in [('Simple RNN', (rnn_preds, rnn_labels)),
                                ('LSTM', (lstm_preds, lstm_labels)),
                                ('GRU', (gru_preds, gru_labels))]:
    fig = plot_confusion_matrix(labels, preds, f'{name} - Confusion Matrix')
    fig.savefig(f'reports/cm_{name.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.close()

if 'BERT (FinBERT)' in all_results:
    fig = plot_confusion_matrix(bert_labels, bert_preds, 'BERT (FinBERT) - Confusion Matrix')
    fig.savefig('reports/cm_bert.png', dpi=150, bbox_inches='tight')
    plt.close()

# Generate PDF report
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# Page 1: Overview and Results
pdf.add_page()
pdf.set_font('Helvetica', 'B', 16)
pdf.cell(0, 10, 'Financial News Sentiment Prediction', ln=True, align='C')
pdf.set_font('Helvetica', '', 11)
pdf.cell(0, 8, 'Comparison Report: RNN vs BERT Architectures', ln=True, align='C')
pdf.ln(5)

# Dataset info
pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Dataset', ln=True)
pdf.set_font('Helvetica', '', 10)
pdf.multi_cell(0, 6, 
    'Twitter Financial News Sentiment (Hugging Face)\n'
    f'Training: {len(train_df)} samples | Validation: {len(val_df)} samples\n'
    'Labels: Bearish (0), Bullish (1), Neutral (2)')
pdf.ln(3)

# Results table
pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Performance Summary', ln=True)
pdf.set_font('Helvetica', 'B', 9)
col_widths = [35, 25, 25, 25, 25, 25, 25]
headers = ['Model', 'Accuracy', 'Macro F1', 'Bear P', 'Bull P', 'Neut P', 'Neut R']
for i, h in enumerate(headers):
    pdf.cell(col_widths[i], 7, h, border=1, align='C')
pdf.ln()

pdf.set_font('Helvetica', '', 9)
for model_name, metrics in all_results.items():
    row_data = [
        model_name[:12],
        f"{metrics['accuracy']:.4f}",
        f"{metrics['macro_f1']:.4f}",
        f"{metrics['class_precision']['Bearish']:.4f}",
        f"{metrics['class_precision']['Bullish']:.4f}",
        f"{metrics['class_precision']['Neutral']:.4f}",
        f"{metrics['class_recall']['Neutral']:.4f}",
    ]
    for i, val in enumerate(row_data):
        pdf.cell(col_widths[i], 7, val, border=1, align='C')
    pdf.ln()

pdf.ln(3)

# Comparison chart
pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Performance Comparison Chart', ln=True)
if os.path.exists('reports/model_comparison.png'):
    pdf.image('reports/model_comparison.png', x=15, w=180)

# Page 2: Analysis
pdf.add_page()
pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Analysis and Findings', ln=True)
pdf.set_font('Helvetica', '', 10)

# RNN comparison
pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 8, '1. RNN Architecture Comparison', ln=True)
pdf.set_font('Helvetica', '', 10)
analysis_text = (
    f'Among the RNN variants, {best_rnn_name} achieved the best performance '
    f'with a Macro F1 score of {best_rnn_f1:.4f}. '
    'LSTM and GRU models generally outperform Simple RNN due to their gating mechanisms '
    'that help capture longer-range dependencies in tweet text. '
    'Simple RNN suffers from vanishing gradient issues, making it harder to learn '
    'meaningful patterns from financial language.'
)
pdf.multi_cell(0, 6, analysis_text)
pdf.ln(3)

# BERT improvement
if 'BERT (FinBERT)' in all_results:
    pdf.set_font('Helvetica', 'B', 11)
    pdf.cell(0, 8, '2. BERT Performance Improvement', ln=True)
    pdf.set_font('Helvetica', '', 10)
    bert_analysis = (
        f'BERT (FinBERT) achieved a Macro F1 of {bert_f1:.4f}, representing a '
        f'{improvement:+.1f}% improvement over the best RNN model. '
        'This improvement stems from BERT\'s pre-trained contextual embeddings '
        'which understand financial terminology and sentiment nuances that '
        'randomly initialized embeddings in RNN models cannot capture. '
        'The transformer attention mechanism also allows BERT to weigh '
        'important tokens regardless of their position in the sequence.'
    )
    pdf.multi_cell(0, 6, bert_analysis)
    pdf.ln(3)

# Key findings
pdf.set_font('Helvetica', 'B', 11)
section_num = '3' if 'BERT (FinBERT)' in all_results else '2'
pdf.cell(0, 8, f'{section_num}. Key Findings', ln=True)
pdf.set_font('Helvetica', '', 10)
findings = (
    '- Neutral class typically has the highest support and is easiest to classify\n'
    '- Bearish and Bullish classes are harder to distinguish due to class imbalance\n'
    '- Early stopping effectively prevents overfitting across all models\n'
    '- Financial domain-specific pre-training (FinBERT) provides significant advantage\n'
    '- All models benefit from proper text preprocessing (URL/mention removal)'
)
pdf.multi_cell(0, 6, findings)

# Save PDF
pdf.output('reports/comparison_report.pdf')
print('Report saved to reports/comparison_report.pdf')

## Summary

This notebook implements a complete financial sentiment classification pipeline:

1. **Data Loading**: Loaded 11,932 finance tweets from Hugging Face (9,938 train / 2,486 val)
2. **Preprocessing**: Cleaned tweets, built vocabulary, created padded tensors
3. **Baseline Models**: Trained Simple RNN, LSTM, and GRU with early stopping
4. **BERT Fine-tuning**: Fine-tuned FinBERT for improved performance
5. **Evaluation**: Compared models using accuracy, macro F1, precision, recall, and confusion matrices
6. **Inference**: Automated script for predicting sentiment of raw financial text
7. **Report**: Generated a 2-page PDF comparison report

Use the Streamlit app (`streamlit run app/streamlit_app.py`) for an interactive sentiment dashboard.

## 10. Streamlit Dashboard & Final Report

### Streamlit App Features
- **Model selector** (sidebar): Switch between RNN and BERT models
- **Model info panel**: Architecture, parameter count, vocab size, max sequence length
- **Text input**: Type any tweet/headline, or use quick-test samples
- **Prediction results**: Color-coded sentiment, confidence %, class probability metrics
- **Bar chart**: Interactive Plotly chart of class probabilities
- **Validation distribution**: Bar chart of class distribution (validation set)

**To launch:**
```bash
cd "Financial news sentiment prediction using DL and BERT"
streamlit run app/streamlit_app.py
```

App runs at [http://localhost:8501](http://localhost:8501).

### Key Observations
- **BERT**: High-confidence, correct predictions for all classes
- **Corrected RNNs**: Produce class-specific predictions after padding fix and retraining
- **Switching models**: Instantly shows the performance gap

### Report & Comparison
- See `reports/Report_and_Comparison.md` for a detailed writeup
- PDF report with charts: `reports/comparison_report.pdf`

**Summary Table:**

| Model           | Accuracy | Macro F1 | Bearish P/R | Bullish P/R | Neutral P/R |
|-----------------|----------|----------|-------------|-------------|-------------|
| Simple RNN      | 72.0%    | 0.632    | 0.45/0.59   | 0.56/0.56   | 0.86/0.80   |
| LSTM            | 78.3%    | 0.709    | 0.55/0.63   | 0.66/0.69   | 0.89/0.84   |
| GRU             | 79.1%    | 0.712    | 0.57/0.60   | 0.67/0.68   | 0.88/0.87   |
| **BERT (FinBERT)** | **82.8%** | **0.760** | 0.80/0.61   | 0.78/0.64   | 0.84/0.93   |

**BERT achieves +6.76% Macro F1 improvement over the corrected best RNN (GRU).**

---

For screenshots and a full UI description, see the report and the app itself.